# Ad Optimization Agent

### Basic dataset analysis

source dataset: https://www.kaggle.com/datasets/manishabhatt22/marketing-campaign-performance-dataset?resource=download

### Load original dataset

In [9]:
import pandas as pd

# Load the dataset
ad_data = pd.read_csv('marketing_campaign_dataset.csv')     

In [10]:
ad_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       200000 non-null  int64  
 1   Company           200000 non-null  object 
 2   Campaign_Type     200000 non-null  object 
 3   Target_Audience   200000 non-null  object 
 4   Duration          200000 non-null  object 
 5   Channel_Used      200000 non-null  object 
 6   Conversion_Rate   200000 non-null  float64
 7   Acquisition_Cost  200000 non-null  object 
 8   ROI               200000 non-null  float64
 9   Location          200000 non-null  object 
 10  Language          200000 non-null  object 
 11  Clicks            200000 non-null  int64  
 12  Impressions       200000 non-null  int64  
 13  Engagement_Score  200000 non-null  int64  
 14  Customer_Segment  200000 non-null  object 
 15  Date              200000 non-null  object 
dtypes: float64(2), int64

### Isolate relevant columns 

In [11]:
ad_data_df = ad_data[['Campaign_ID', 'Target_Audience', 'Duration', 'Channel_Used', 'Conversion_Rate', 'ROI', 'Clicks', 'Impressions', 'Date']]

In [12]:
ad_data_df['Date']= pd.to_datetime(ad_data_df['Date'])


/var/folders/36/fd3r2w554k11df_csyq6r1sc0000gn/T/ipykernel_4736/1502540321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ad_data_df['Date']= pd.to_datetime(ad_data_df['Date'])


In [13]:
min_date = ad_data['Date'].min()
max_date = ad_data['Date'].max()
print(f"Date range: {min_date} to {max_date}")

Date range: 2021-01-01 to 2021-12-31


### Isolate June data and understand Channels

In [14]:
ad_data_june = ad_data_df[ad_data_df['Date'].dt.month == 6]
ad_data_june['Channel_Used'].unique()

array(['Website', 'Instagram', 'Facebook', 'Google Ads', 'YouTube',
       'Email'], dtype=object)

In [15]:
ad_data = ad_data_june[ad_data_june['Channel_Used'].isin(['Instagram', 'Google Ads', 'YouTube'])]
ad_data['Channel_Used'].unique()

array(['Instagram', 'Google Ads', 'YouTube'], dtype=object)

### Create smaller dataset with 3 chosen channels for the month of June

In [39]:
ad_data = ad_data_june[ad_data_june['Channel_Used'].isin(['Instagram', 'Google Ads', 'YouTube'])]
ad_data.to_csv('ad_data_june.csv', index=False)

In [40]:
ad_data = pd.read_csv('ad_data_june.csv', parse_dates=['Date'])


In [41]:
ad_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8200 entries, 0 to 8199
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Campaign_ID      8200 non-null   int64         
 1   Target_Audience  8200 non-null   object        
 2   Duration         8200 non-null   object        
 3   Channel_Used     8200 non-null   object        
 4   Conversion_Rate  8200 non-null   float64       
 5   ROI              8200 non-null   float64       
 6   Clicks           8200 non-null   int64         
 7   Impressions      8200 non-null   int64         
 8   Date             8200 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(3), object(3)
memory usage: 576.7+ KB


In [42]:
# Average Conversion Rate and ROI per Channel
avg_per_channel = ad_data.groupby('Channel_Used').agg(
    avg_conversion_rate=('Conversion_Rate', 'mean'),
    avg_roi=('ROI', 'mean')
).round(4).sort_values('avg_roi', ascending=False).reset_index()

print(avg_per_channel)

  Channel_Used  avg_conversion_rate  avg_roi
0    Instagram               0.0790   5.0042
1      YouTube               0.0792   4.9741
2   Google Ads               0.0806   4.9719


A stateful AI agent built with **LangGraph** that reads June 2021 campaign data from a CSV, allocates a $1,000 daily budget across three paid channels (Instagram, Google Ads, YouTube), and logs every decision with rationale.

**Each notebook run resets to the 34/33/33 baseline** and runs the agent once. The internal LangGraph loop handles all reasoning and tool calls until the agent decides it is done.

### Agent Loop
```
load_csv_data → agent_reasoning ──→ execute_tools → agent_reasoning → END
                               ↘ END (direct)
```

### Guardrails (auto-enforced in tools)
| Rule | Value |
|------|-------|
| Daily budget | $1,000 across 3 channels |
| Starting split | 34% Google Ads, 33% Instagram, 33% YouTube |
| Max change per run | ±20% per channel |
| Minimum floor | 5% (never 0%) |
| Max consecutive pause | 2 days |

**Primary metric:** Conversion Rate (CVR) | **Secondary:** ROI


## 1. Install Dependencies

In [16]:
%pip install langgraph langchain langchain_openai pydantic pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Imports, State & Global Budget State

In [17]:
import os
import json
import operator
import pandas as pd
from typing import TypedDict, Annotated, List, Dict
from dotenv import load_dotenv

from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

load_dotenv()

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Agent State
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    channel_data: Dict          # aggregated June metrics per channel
    channel_allocations: Dict   # {channel: float pct}
    next_action: str

# Constants
PAID_CHANNELS = ["Google Ads", "Instagram", "YouTube"]
DAILY_BUDGET  = 1000.0

# Baseline split — always used at the start of every notebook run
DEFAULT_ALLOCATIONS = {"Google Ads": 34.0, "Instagram": 33.0, "YouTube": 33.0}

# Global mutable budget state (shared across all tool calls in one run)
BUDGET_STATE: Dict = {
    "allocations": {},
    "log":         [],
    "paused_days": {}
}


def initialize_budget():
    """Reset to the 34/33/33 baseline so every notebook run starts fresh."""
    BUDGET_STATE["allocations"] = dict(DEFAULT_ALLOCATIONS)
    BUDGET_STATE["paused_days"] = {ch: 0 for ch in PAID_CHANNELS}
    BUDGET_STATE["log"]         = []
    print("Reset to baseline: 34% Google Ads / 33% Instagram / 33% YouTube")


print("Imports OK.")


Imports OK.


## 3. Tools

Three tools the LLM can invoke. Guardrails are enforced inside each tool and recorded in the log.

| Tool | Purpose |
|------|---------|
| `allocate_budget` | Shift % to a channel (±20% cap, 5% floor) |
| `pause_channel` | Flag a channel as paused (max 2 consecutive days) |
| `request_new_creatives` | Request new ad assets for a channel |

In [18]:
@tool
def allocate_budget(channel: str, new_pct: float, reason: str) -> str:
    """
    Shift the daily budget percentage for `channel` to `new_pct`.
    Remaining % is distributed equally among the other three channels.
    Guardrails: max ±20% change; 5% minimum floor.
    """
    if channel not in PAID_CHANNELS:
        return f"Unknown channel '{channel}'. Valid: {PAID_CHANNELS}"

    current_pct = BUDGET_STATE["allocations"][channel]
    notes = []

    delta = new_pct - current_pct
    if delta > 20:
        new_pct = current_pct + 20
        notes.append(f"capped at +20% (requested {delta:+.1f}%)")
    elif delta < -20:
        new_pct = current_pct - 20
        notes.append(f"capped at -20% (requested {delta:+.1f}%)")

    if new_pct < 5.0:
        new_pct = 5.0
        notes.append("floored at 5%")

    others = [ch for ch in PAID_CHANNELS if ch != channel]
    per_other = round((100.0 - new_pct) / len(others), 4)
    for ch in others:
        BUDGET_STATE["allocations"][ch] = per_other
    BUDGET_STATE["allocations"][channel] = new_pct

    daily_amount = round(new_pct / 100 * DAILY_BUDGET, 2)
    BUDGET_STATE["log"].append({
        "action":          "allocate_budget",
        "channel":         channel,
        "from_pct":        current_pct,
        "to_pct":          new_pct,
        "daily_amount":    daily_amount,
        "reason":          reason,
        "guardrail_notes": "; ".join(notes) if notes else "none"
    })

    msg = (f"Budget for {channel}: {current_pct:.1f}% → {new_pct:.1f}% "
           f"(${daily_amount:.2f}/day). Reason: {reason}")
    if notes:
        msg += f" | Guardrails: {'; '.join(notes)}"
    return msg


@tool
def pause_channel(channel: str, reason: str) -> str:
    """
    Flag `channel` as paused for underperformance.
    A channel cannot be paused for more than 2 consecutive days.
    """
    if channel not in PAID_CHANNELS:
        return f"Unknown channel '{channel}'."

    consecutive = BUDGET_STATE["paused_days"].get(channel, 0)
    notes = []

    if consecutive >= 2:
        notes.append(f"pause blocked — already paused {consecutive} consecutive day(s)")
        status = "BLOCKED"
    else:
        BUDGET_STATE["paused_days"][channel] = consecutive + 1
        status = "PAUSED"

    BUDGET_STATE["log"].append({
        "action":           "pause_channel",
        "channel":          channel,
        "status":           status,
        "consecutive_days": BUDGET_STATE["paused_days"][channel],
        "reason":           reason,
        "guardrail_notes":  "; ".join(notes) if notes else "none"
    })

    msg = f"Channel {channel} {status}. Reason: {reason}"
    if notes:
        msg += f" | {'; '.join(notes)}"
    return msg


@tool
def request_new_creatives(channel: str, reason: str) -> str:
    """Request new creative assets for `channel` to improve engagement."""
    if channel not in PAID_CHANNELS:
        return f"Unknown channel '{channel}'."

    BUDGET_STATE["log"].append({
        "action":          "request_new_creatives",
        "channel":         channel,
        "reason":          reason,
        "guardrail_notes": "none"
    })
    return f"New creative assets requested for {channel}. Reason: {reason}"


ad_optimization_tools = [allocate_budget, pause_channel, request_new_creatives]
print("Tools defined.")

Tools defined.


## 4. Graph Nodes

In [19]:
def load_csv_data(state: AgentState) -> dict:
    """
    Node 1: reads ad_data_june.csv (pre-filtered to 3 paid channels, June 2021),
    aggregates per-channel metrics, and primes the LLM with a performance summary.
    """
    print("--- LOADING CSV DATA ---")

    df = pd.read_csv("ad_data_june.csv", parse_dates=["Date"])

    agg = df.groupby("Channel_Used").agg(
        avg_cvr=("Conversion_Rate",  "mean"),
        avg_roi=("ROI",              "mean"),
        total_clicks=("Clicks",      "sum"),
        total_impressions=("Impressions", "sum"),
        row_count=("Campaign_ID",    "count")
    ).reset_index()
    agg["ctr"] = agg["total_clicks"] / agg["total_impressions"]

    channel_data = {}
    for _, row in agg.iterrows():
        channel_data[row["Channel_Used"]] = {
            "avg_cvr":   round(row["avg_cvr"],  4),
            "avg_roi":   round(row["avg_roi"],  4),
            "ctr":       round(row["ctr"],       4),
            "row_count": int(row["row_count"])
        }

    lines = ["June 2021 Channel Performance (paid channels only):\n"]
    for ch, m in channel_data.items():
        lines.append(
            f"  {ch}: CVR={m['avg_cvr']:.2%}  ROI={m['avg_roi']:.2f}  "
            f"CTR={m['ctr']:.2%}  (n={m['row_count']})"
        )

    lines.append("\nCurrent budget allocations (% of $1,000/day):")
    for ch, pct in BUDGET_STATE["allocations"].items():
        lines.append(f"  {ch}: {pct:.1f}% = ${pct/100*DAILY_BUDGET:.2f}")

    lines.append(
        "\nTask: analyse the channel performance above. "
        "Use tools to shift budget toward top performers (by CVR then ROI), "
        "pause poor performers, or request new creatives where engagement is low. "
        "After using all necessary tools, provide a concise final summary."
    )

    prompt = "\n".join(lines)
    print(prompt)

    return {
        "messages":            [HumanMessage(content=prompt)],
        "channel_data":        channel_data,
        "channel_allocations": dict(BUDGET_STATE["allocations"]),
        "next_action":         "start"
    }


def agent_reasoning(state: AgentState) -> dict:
    """Node 2: LLM analyses performance and selects a tool or finishes."""
    print("--- AGENT REASONING ---")

    llm_with_tools = llm.bind_tools(ad_optimization_tools)

    system_prompt = (
        "You are an expert Ad Optimization Agent managing a $1,000 daily budget "
        "across three paid channels: Google Ads, Instagram, and YouTube. "
        "Primary metric: Conversion Rate (CVR). Secondary: ROI. "
        "Shift budget toward the highest CVR channels, pause channels with poor CVR and ROI, "
        "and request new creatives where engagement metrics are weak. "
        "Guardrails (+-20% cap, 5% floor, 2-day pause limit) are enforced automatically. "
        "After issuing all necessary tool calls, produce a concise final summary "
        "of your decisions and expected outcomes — do NOT call another tool."
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("placeholder", "{messages}")
    ])

    response = (prompt | llm_with_tools).invoke(state)

    if response.tool_calls:
        return {"messages": [response], "next_action": "call_tool"}
    return {"messages": [response], "next_action": "FINISH"}


def execute_tools(state: AgentState) -> dict:
    """Node 3: runs the tool(s) selected by agent_reasoning."""
    print("--- EXECUTING TOOL ---")

    tool_map   = {t.name: t for t in ad_optimization_tools}
    tool_calls = state["messages"][-1].tool_calls
    results    = []

    for call in tool_calls:
        fn = tool_map.get(call["name"])
        if fn:
            try:
                result = fn.invoke(call["args"])
            except Exception as e:
                result = f"Error in {call['name']}: {e}"
        else:
            result = f"Tool '{call['name']}' not found."

        print(f"  [{call['name']}] -> {result}")
        results.append(ToolMessage(content=result, tool_call_id=call["id"]))

    return {
        "messages":            results,
        "channel_allocations": dict(BUDGET_STATE["allocations"])
    }


def decide_next_step(state: AgentState) -> str:
    """Router: tool call -> execute_tools; otherwise -> END."""
    return "call_tool" if state.get("next_action") == "call_tool" else "end"


print("Nodes defined.")


Nodes defined.


## 5. Build & Compile the LangGraph

In [20]:
workflow = StateGraph(AgentState)

workflow.add_node("load_csv_data",   load_csv_data)
workflow.add_node("agent_reasoning", agent_reasoning)
workflow.add_node("execute_tools",   execute_tools)

workflow.set_entry_point("load_csv_data")
workflow.add_edge("load_csv_data", "agent_reasoning")

workflow.add_conditional_edges(
    "agent_reasoning",
    decide_next_step,
    {"call_tool": "execute_tools", "end": END}
)

# After tools run, return to agent_reasoning for follow-up or final summary
workflow.add_edge("execute_tools", "agent_reasoning")

app = workflow.compile()
print("Graph compiled.")

Graph compiled.


## 6. Run the Agent

In [21]:
initialize_budget()

print("=" * 60)
print("STARTING AD OPTIMIZATION AGENT RUN")
print("=" * 60)

final_state = app.invoke(
    {
        "messages":            [],
        "channel_data":        {},
        "channel_allocations": {},
        "next_action":         "start"
    },
    config={"recursion_limit": 25}
)

print("\n" + "=" * 60)
print("AGENT FINAL SUMMARY")
print("=" * 60)
print(final_state["messages"][-1].content)


Reset to baseline: 34% Google Ads / 33% Instagram / 33% YouTube
STARTING AD OPTIMIZATION AGENT RUN
--- LOADING CSV DATA ---
June 2021 Channel Performance (paid channels only):

  Google Ads: CVR=8.06%  ROI=4.97  CTR=9.79%  (n=2721)
  Instagram: CVR=10.81%  ROI=6.13  CTR=12.88%  (n=2722)
  YouTube: CVR=4.41%  ROI=3.25  CTR=5.96%  (n=2757)

Current budget allocations (% of $1,000/day):
  Google Ads: 34.0% = $340.00
  Instagram: 33.0% = $330.00
  YouTube: 33.0% = $330.00

Task: analyse the channel performance above. Use tools to shift budget toward top performers (by CVR then ROI), pause poor performers, or request new creatives where engagement is low. After using all necessary tools, provide a concise final summary.
--- AGENT REASONING ---
--- EXECUTING TOOL ---
  [allocate_budget] -> Budget for Instagram: 33.0% → 40.0% ($400.00/day). Reason: Highest CVR and ROI, shifting budget to maximize performance.
  [allocate_budget] -> Budget for Google Ads: 30.0% → 40.0% ($400.00/day). Reason: S

## 7. Evaluation & Decision Log

Print the final budget split, the full decision log (with guardrail notes), and a per-channel performance snapshot.

In [22]:
# Baseline: equal split (33.33% each)
BASELINE = {ch: round(100 / len(PAID_CHANNELS), 2) for ch in PAID_CHANNELS}

print("=" * 60)
print("FINAL BUDGET ALLOCATIONS vs EQUAL-SPLIT BASELINE")
print("=" * 60)
print(f"  {'Channel':15s}  {'Baseline %':>10}  {'Optimized %':>12}  {'Daily $':>10}")
print(f"  {'-'*15}  {'-'*10}  {'-'*12}  {'-'*10}")
total = 0.0
for ch in PAID_CHANNELS:
    pct   = BUDGET_STATE["allocations"].get(ch, BASELINE[ch])
    base  = BASELINE[ch]
    daily = pct / 100 * DAILY_BUDGET
    total += daily
    diff  = pct - base
    arrow = f"(+{diff:.1f}%)" if diff > 0 else f"({diff:.1f}%)" if diff < 0 else "(no change)"
    print(f"  {ch:15s}  {base:>9.1f}%  {pct:>10.1f}%  ${daily:>8.2f}  {arrow}")
print(f"\n  Total daily spend: ${total:.2f}")

print("\n" + "=" * 60)
print("DECISION LOG")
print("=" * 60)
if not BUDGET_STATE["log"]:
    print("  (no actions taken)")
for i, entry in enumerate(BUDGET_STATE["log"], 1):
    print(f"\n  [{i}] Action : {entry['action']}")
    for k, v in entry.items():
        if k != "action":
            print(f"      {k:20s}: {v}")

print("\n" + "=" * 60)
print("CHANNEL PERFORMANCE SNAPSHOT  (June 2021)")
print("=" * 60)
print(f"  {'Channel':15s}  {'CVR':>7}  {'ROI':>6}  {'CTR':>7}  {'Final %':>8}  {'Baseline %':>10}")
print(f"  {'-'*15}  {'-'*7}  {'-'*6}  {'-'*7}  {'-'*8}  {'-'*10}")
for ch, m in final_state.get("channel_data", {}).items():
    final_pct = BUDGET_STATE["allocations"].get(ch, BASELINE.get(ch, 33.33))
    base_pct  = BASELINE.get(ch, 33.33)
    print(
        f"  {ch:15s}  {m['avg_cvr']:>6.2%}  {m['avg_roi']:>6.2f}  "
        f"{m['ctr']:>6.2%}  {final_pct:>7.1f}%  {base_pct:>9.1f}%"
    )

print("\n" + "=" * 60)
print("EVALUATION NOTES")
print("=" * 60)
print("  Baseline: equal split ($333.33/day per channel).")
print("  The agent allocates more to channels with higher CVR and ROI,")
print("  and reduces or pauses those with weaker performance.")
print("  Primary metric: CVR (Conversion Rate). Secondary: ROI.")


FINAL BUDGET ALLOCATIONS vs EQUAL-SPLIT BASELINE
  Channel          Baseline %   Optimized %     Daily $
  ---------------  ----------  ------------  ----------
  Google Ads            33.3%        40.0%  $  400.00  (+6.7%)
  Instagram             33.3%        30.0%  $  300.00  (-3.3%)
  YouTube               33.3%        30.0%  $  300.00  (-3.3%)

  Total daily spend: $1000.00

DECISION LOG

  [1] Action : allocate_budget
      channel             : Instagram
      from_pct            : 33.0
      to_pct              : 40.0
      daily_amount        : 400.0
      reason              : Highest CVR and ROI, shifting budget to maximize performance.
      guardrail_notes     : none

  [2] Action : allocate_budget
      channel             : Google Ads
      from_pct            : 30.0
      to_pct              : 40.0
      daily_amount        : 400.0
      reason              : Second highest CVR and ROI, shifting budget to maximize performance.
      guardrail_notes     : none

  [3] Acti